In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)
print(tf.config.list_physical_devices("GPU"))

In [ ]:
(full_train_images, full_train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
full_train_images = full_train_images.astype("float32") / 255.0
test_images = test_images.astype("float32") / 255.0
full_train_labels = full_train_labels.flatten()
test_labels = test_labels.flatten()

train_images, val_images = full_train_images[:45000], full_train_images[45000:]
train_labels, val_labels = full_train_labels[:45000], full_train_labels[45000:]
print(train_images.shape, val_images.shape, test_images.shape)

In [ ]:
def build_cnn():
    return keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax")
    ])

keras.utils.set_random_seed(7)
reference_model = build_cnn()
starting_weights = reference_model.get_weights()
reference_model.summary()

class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.epoch_times = []
    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.time()
    def on_epoch_end(self, epoch, logs=None):
        self.epoch_times.append(time.time() - self.epoch_start)

In [ ]:
optimizer_settings = {
    "SGD": lambda: keras.optimizers.SGD(learning_rate=0.01),
    "SGD + momentum": lambda: keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSProp": lambda: keras.optimizers.RMSprop(learning_rate=1e-3),
    "Adam": lambda: keras.optimizers.Adam(learning_rate=1e-3)
}

optimizer_histories = {}
optimizer_times = {}
optimizer_models = {}
for name, make_optimizer in optimizer_settings.items():
    keras.utils.set_random_seed(7)
    model = build_cnn()
    model.set_weights(starting_weights)
    model.compile(optimizer=make_optimizer(), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    timer = EpochTimer()
    history = model.fit(train_images, train_labels, validation_data=(val_images, val_labels), epochs=20, batch_size=128, verbose=0, callbacks=[timer])
    optimizer_histories[name] = history.history
    optimizer_times[name] = np.mean(timer.epoch_times[1:])
    optimizer_models[name] = model
    print(f"{name:<15} final train loss {history.history['loss'][-1]:.4f}  best val acc {max(history.history['val_accuracy']):.4f}  {optimizer_times[name]:.1f}s per epoch")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plot_layout = [("loss", "Training loss"), ("val_loss", "Validation loss"), ("accuracy", "Training accuracy"), ("val_accuracy", "Validation accuracy")]
for axis, (metric, title) in zip(axes.flatten(), plot_layout):
    for name, history in optimizer_histories.items():
        axis.plot(range(1, len(history[metric]) + 1), history[metric], marker="o", markersize=3, label=name)
    axis.set_title(title)
    axis.set_xlabel("epoch")
    axis.legend()
plt.tight_layout()
plt.show()

In [ ]:
summary_rows = []
for name, history in optimizer_histories.items():
    test_loss, test_accuracy = optimizer_models[name].evaluate(test_images, test_labels, verbose=0)
    reached = [epoch for epoch, value in enumerate(history["val_accuracy"], start=1) if value >= 0.6]
    summary_rows.append({
        "optimizer": name,
        "final train loss": history["loss"][-1],
        "lowest val loss": min(history["val_loss"]),
        "epoch of lowest val loss": int(np.argmin(history["val_loss"])) + 1,
        "best val accuracy": max(history["val_accuracy"]),
        "epochs to 60% val accuracy": reached[0] if reached else None,
        "test accuracy": test_accuracy,
        "seconds per epoch": optimizer_times[name]
    })
optimizer_summary = pd.DataFrame(summary_rows).set_index("optimizer").round(4)
display(optimizer_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
optimizer_summary["test accuracy"].plot(kind="bar", ax=axes[0], rot=0, color="teal", title="Test accuracy")
optimizer_summary["lowest val loss"].plot(kind="bar", ax=axes[1], rot=0, color="coral", title="Lowest validation loss")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
for name, history in optimizer_histories.items():
    gap = np.array(history["val_loss"]) - np.array(history["loss"])
    plt.plot(range(1, len(gap) + 1), gap, label=name)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Overfitting gap (validation loss minus training loss)")
plt.xlabel("epoch")
plt.legend()
plt.show()

In [ ]:
optimizer_factories = {
    "SGD": lambda learning_rate: keras.optimizers.SGD(learning_rate=learning_rate),
    "SGD + momentum": lambda learning_rate: keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9),
    "RMSProp": lambda learning_rate: keras.optimizers.RMSprop(learning_rate=learning_rate),
    "Adam": lambda learning_rate: keras.optimizers.Adam(learning_rate=learning_rate)
}
learning_rates = [1e-4, 1e-3, 1e-2, 1e-1]
sensitivity_rows = []

for name, make_optimizer in optimizer_factories.items():
    for learning_rate in learning_rates:
        keras.utils.set_random_seed(7)
        model = build_cnn()
        model.set_weights(starting_weights)
        model.compile(optimizer=make_optimizer(learning_rate), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        history = model.fit(train_images, train_labels, validation_data=(val_images, val_labels), epochs=4, batch_size=128, verbose=0)
        sensitivity_rows.append({"optimizer": name, "learning rate": learning_rate, "val accuracy": max(history.history["val_accuracy"])})
        print(f"{name:<15} lr {learning_rate:<7} best val acc {sensitivity_rows[-1]['val accuracy']:.4f}")

In [ ]:
sensitivity_frame = pd.DataFrame(sensitivity_rows)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for name, group in sensitivity_frame.groupby("optimizer"):
    axes[0].plot(group["learning rate"], group["val accuracy"], marker="o", label=name)
axes[0].set_xscale("log")
axes[0].set_xlabel("learning rate")
axes[0].set_ylabel("best validation accuracy after 4 epochs")
axes[0].set_title("Learning rate sensitivity")
axes[0].legend()
sns.heatmap(sensitivity_frame.pivot(index="optimizer", columns="learning rate", values="val accuracy"), annot=True, fmt=".3f", cmap="magma", ax=axes[1])
axes[1].set_title("Validation accuracy grid")
plt.show()